### Практика: параметрически эффективное дообучение (PEFT)

В этом ноутбуке ты научишься дообучать большие языковые модели при ограниченном объёме памяти GPU.

> Неофициальный русский перевод [оригинального семинара](https://github.com/yandexdataschool/nlp_course/blob/2026/week08_finetuning/seminar.ipynb). Код не изменён.

In [1]:
%pip install --upgrade peft bitsandbytes

import torch
import torch.nn as nn
import torch.nn.functional as F

import transformers
from tqdm.auto import tqdm, trange
assert torch.cuda.is_available(), "you need cuda for this part"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
model_name = 'unsloth/Qwen3-8B-Base-bnb-4bit'
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token_id = tokenizer.eos_token_id

# основные веса загружены с точностью 4 бита, но LoRA-адаптеры всё равно можно обучать
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, device_map='auto', low_cpu_mem_usage=True, dtype=torch.float32,
    quantization_config=transformers.BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4"),
)
for param in model.parameters():
    param.requires_grad=False
model.gradient_checkpointing_enable()  # only store a small subset of activations, re-compute the rest.
model.enable_input_require_grads()     # override an implementation quirk in gradient checkpoints that disables backprop unless inputs require grad
# подробнее о gradient checkpointing: https://pytorch.org/docs/stable/checkpoint.html https://arxiv.org/abs/1604.06174

/usr/local/lib/python3.13/dist-packages/transformers/quantizers/auto.py:279: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

### Prompt tuning: история про лису (1 балл)

![img](https://i.imgur.com/Ux3qQAu.png) (источник: theodd1souts.fandom.com)

In [4]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

for i in range(10):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0].cpu().numpy().tolist()))


Output: A quick brown fox jumps over the lazy dog. The quick brown fox


Какая откровенная ложь! Эта лиса уверяет, что на самом деле **не перепрыгивала через ленивую собаку**. Нет уж — лиса просто занималась своими делами.

**Твоя задача:** обучить модель говорить «правду»: сегодня никто не перепрыгивал через собаку.

In [5]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
outputs = model(**batch)

next_word_logits = outputs.logits[:, :-1]
true_next_tokens = batch['input_ids'][:, 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))

print("Loss:", loss)

Loss: tensor(3.5625, device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<NllLossBackward0>)


Но мы не можем обучать всю модель: одни только градиенты в float32 потребовали бы 28 ГБ. Вместо этого используем [prompt tuning](https://arxiv.org/abs/2104.08691).

![img](https://i.imgur.com/VwNNKnb.png)

In [ ]:
class WordEmbeddingsWithLearnedPrompts(nn.Module):
    """
    To perform prompt tuning, you will need to replace model's original word embeddings with a layer - THIS layer
     - that inserts trainable prompts instead of the first N token embeddings. """

    def __init__(self, word_embeddings: nn.Embedding, num_prompts: int):
        super().__init__()
        self.original_word_embeddings = word_embeddings
        self.num_prompts = num_prompts
        self.learnable_prompts = nn.Parameter(
            torch.randn(1, num_prompts, word_embeddings.embedding_dim), requires_grad=True)

    def forward(self, input_ids: torch.LongTensor):
        # форма input_ids: [batch_size, seq length]
        assert input_ids.dtype == torch.int64
        assert input_ids.shape[1] > self.num_prompts
        assert torch.all(input_ids[:, :self.num_prompts] == tokenizer.pad_token_id).item(), "don't forget to prepend several BOS tokens to input_ids"

        # Задание: получи эмбеддинги input_ids, заменив первые :num_prompts: токенов на self.learnable_prompts
        # Это нужно, потому что в начале мы добавим :num_prompts: padding-токенов

        # В результате должен получиться вектор эмбеддинга для каждого токена input_ids,
        # причём первые :num_prompts: векторов должны быть равны learnable_prompts;
        # все остальные векторы вычисляются обычным способом
        # С обучаемыми параметрами используй torch.cat, а не присваивание по индексам

        <YOUR CODE HERE>

        return your_outputs_with_prompts_as_per_instructions_above

In [ ]:
num_prompts = 16
test_emb_layer = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)
test_input_ids = tokenizer("a cat say on a may", return_tensors='pt')['input_ids'].to(device)

space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
test_inputs_with_prompts = torch.cat([space_for_prompts, test_input_ids], dim=1)

with torch.cuda.amp.autocast():
  test_prompt_embeddings = test_emb_layer(test_inputs_with_prompts)

assert test_prompt_embeddings.shape[:2] == test_inputs_with_prompts.shape
assert test_prompt_embeddings.shape[-1] == model.config.hidden_size
assert torch.allclose(test_prompt_embeddings[:, :num_prompts], test_emb_layer.learnable_prompts.float())
assert torch.allclose(test_prompt_embeddings[:, num_prompts:], model.model.embed_tokens(test_input_ids).float())
print("Looks legit!")

Looks legit!


**Теперь, когда всё работает,** внедрим обучаемые промпты в основную модель и обучим её правильному рассказу о лисе.

In [ ]:
assert isinstance(model.model.embed_tokens, nn.Embedding), "you have already replaced the embedding layer. If the replacement is broken, please reload the model"

model.model.embed_tokens = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)

opt = torch.optim.Adam([model.model.embed_tokens.learnable_prompts], lr=0.01)

In [ ]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)

outputs = model(**batch)
next_word_logits = outputs.logits[:, num_prompts : -1, :]
true_next_tokens = batch['input_ids'][:, num_prompts + 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))
print("Loss:", loss)


raise NotImplemented("Your task: iteratively train the model to reduce loss using the optimizer (opt)")


assert loss.item() <= 0.1
print("Good job!")

In [ ]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)


for i in range(15):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0, num_prompts:].cpu().numpy().tolist()))

# Если всё верно, модель будет отрицать, что лиса перепрыгнула через ленивую собаку

### Использование Hugging Face PEFT

Библиотека [`peft`](https://huggingface.co/docs/peft/index) — спутник Transformers, предоставляющий общий интерфейс для **параметрически эффективных методов дообучения**: prompt tuning, prefix tuning и различных адаптеров.

In [ ]:
import peft
assert isinstance(model.model.embed_tokens, nn.Embedding), "please reload the model"

peft_config = peft.PromptTuningConfig(task_type=peft.TaskType.CAUSAL_LM, num_virtual_tokens=16)
model = peft.get_peft_model(model, peft_config)  # note: for most peft methods, this line also modifies model in-place
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Total parameters (excluding quantization):", sum(p.numel() for p in model.parameters()))

Trainable parameters: 65536
Total parameters (excluding quantization): 3500478464


In [ ]:
# Задание: обучи модель в обёртке PEFT до next-token loss < 0.1, используя PEFT
# Теперь PAD-токены спереди не нужны, но первые :num_virtual_tokens: логитов следует пропустить.
# В конце сгенерируй предложение и проверь, что модель выучила нужный ответ.

In [ ]:
# Организуй код как удобно, главное — сохраняй читаемость.

### Параметрически эффективное дообучение через LoRA (1 балл)

Для более серьёзных задач применяют низкоранговые адаптеры из [статьи LoRA](https://arxiv.org/pdf/2106.09685.pdf).

Идея: добавить низкоранговые адаптеры **параллельно существующим линейным слоям**:

<center><img src="https://i.imgur.com/6bQLNiG.png" width=240px></center>

В оригинальной статье адаптеры добавляли только к проекциям внимания. [Позднейшие работы](https://arxiv.org/abs/2305.14314) показывают, что дообучать FFN тоже полезно. Но прежде чем обучать сеть, **реализуй базовый слой LoRA**.

In [2]:
import peft  # v-- re-load the notebook to remove any previous PEFT tuners, run cell 1 and this
model_name = 'unsloth/Qwen3-8B-Base-bnb-4bit'
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token_id = tokenizer.eos_token_id
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, device_map='auto', low_cpu_mem_usage=True, dtype=torch.float32,
    quantization_config=transformers.BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4"),
).to(device)
model = peft.prepare_model_for_kbit_training(model, use_gradient_checkpointing=True) # < checkpoints and input grads

/usr/local/lib/python3.13/dist-packages/transformers/quantizers/auto.py:279: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

In [ ]:
class LoRALayer(nn.Module):
    """Wraps a linear layer with LoRA-like adapter. Wraps an existing OPT linear layer"""
    def __init__(self, module: nn.Linear, rank: int):
        super().__init__()
        self.module = module  # pre-trained (frozen) linear layer
        self.adapter_A = nn.Parameter(torch.empty(module.in_features, rank, device=module.weight.device))
        nn.init.kaiming_uniform_(self.adapter_A, a=5 ** 0.5)
        self.adapter_B = nn.Parameter(torch.zeros(rank, module.out_features, device=module.weight.device))

    def forward(self, input):
        # Примени self.module и адаптер LoRA, верни сумму их выходов
        <YOUR CODE HERE>
        return <YOUR CODE: OUTPUT WITH ADAPTER>

In [7]:
# проверь реализацию
test_linear = nn.Linear(128, 128)
test_linear.weight.data[...] = torch.eye(128)
test_adapter = LoRALayer(test_linear, rank=8)

assert torch.allclose(test_adapter(torch.ones(1, 1, 128)), test_linear.bias + 1), "please check your forward pass"

test_adapter.adapter_A.data[...] = torch.linspace(0.1, -0.5, 128 * 8).view(128, 8)
test_adapter.adapter_B.data[...] = torch.linspace(0.5, -0.1, 128 * 8).view(8, 128)
test_linear.bias.data[...] = torch.linspace(1., -1., 128)

dummy_loss = F.mse_loss(test_adapter(torch.ones(1, 128) / 128).squeeze(), torch.linspace(-1, 1, 128))
assert torch.allclose(dummy_loss, torch.tensor(1.3711389), rtol=0, atol=1e-4)
dummy_loss.backward()
assert all(w.grad is not None for w in [test_adapter.adapter_A, test_adapter.adapter_B]), "some adapter weights have no grad"
assert torch.allclose(test_adapter.adapter_A.grad.sum(), torch.tensor(-0.60158), rtol=0, atol=1e-4), "bad grad w.r.t. A"
assert torch.allclose(test_adapter.adapter_B.grad.sum(), torch.tensor(0.9931), rtol=0, atol=1e-4), "bad grad w.r.t. B"
# Неверный градиент означает отличие от статьи LoRA или нарушение совместимости с autograd (например, no_grad)
del dummy_loss, test_linear, test_adapter
print("All tests passed!")

All tests passed!


### Применяем LoRA к модели

Код ниже добавляет LoRA-адаптеры к линейным проекциям Q/K/V в блоках attention. Можно изменять и другие слои:

- `self_attn.o_proj` — выходная проекция внимания.
- `mlp.up_proj`, `mlp.gate_proj`, `mlp.down_proj` — полносвязные блоки трансформера.
- `lm_head` — выходная голова языковой модели.

**Примечание:** домашнее задание находится ниже.

In [ ]:
lora_rank = 8

for name, module in model.model.layers.named_modules():
    if 'DecoderLayer' in repr(type(module)):
        module.self_attn.q_proj = LoRALayer(module.self_attn.q_proj, rank=lora_rank).to(device)
        module.self_attn.k_proj = LoRALayer(module.self_attn.k_proj, rank=lora_rank).to(device)
        module.self_attn.v_proj = LoRALayer(module.self_attn.v_proj, rank=lora_rank).to(device)

assert sum(isinstance(module, LoRALayer) for module in model.modules()) > 0, "Did not add any LoRA layers!"

In [ ]:
batch = tokenizer("This model wants to share its greatest secret:", return_tensors='pt', return_token_type_ids=False).to(device)
# проверь один шаг обучения и убедись, что градиенты осмысленны
with torch.cuda.amp.autocast(dtype=torch.float32):
    out = model.forward(**batch)
    (out.logits.norm() / 100).backward()

for i, module in enumerate(model.modules()):
    if isinstance(module, LoRALayer):
        assert module.adapter_B.grad is not None
        assert module.adapter_B.grad.norm().item() > 0

model.zero_grad(set_to_none=True)
print("Grad check successful, well done!")

Grad check successful, well done!


### (Пример) Обучение модели

Ниже показан пример обучения LoRA-адаптеров на игрушечном датасете. Позже тебе нужно будет выполнить **похожее задание**.

**Примечание:** прокрути дальше до домашнего задания.

In [ ]:
# проверяем обучаемость модели; для полноценного обучения измени max_steps
import datasets
data = datasets.load_dataset("Abirate/english_quotes", split="train[:32]") # 32 lines
data = data.map(lambda samples: tokenizer(samples['quote']), batched=True)
model._hf_peft_config_loaded = True  # silence a warning from HF trainer

trainer = transformers.Trainer(
    model=model, train_dataset=data,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=1,
        # если нужен больший эффективный батч, увеличь gradient_accumulation_steps
        warmup_steps=250, max_steps=100, learning_rate=2e-4, fp16=True,
        logging_steps=1, output_dir='outputs', report_to="tensorboard"),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
# Если появятся предупреждения про кеш, установи model.config.use_cache = False; перед инференсом включи обратно!

trainer.train()

# Внимание: это всего лишь пример, не обязательно дожидаться завершения progressbar :)

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,1.289705
2,0.375477
3,1.699360
4,1.195340
5,0.786238
6,1.737145
7,1.637755
8,1.208393
9,0.634554
10,1.270690


Step,Training Loss
1,1.289705
2,0.375477
3,1.699360
4,1.195340
5,0.786238
6,1.737145
7,1.637755
8,1.208393
9,0.634554
10,1.270690


### Финальное задание: действительно дообучи модель (3 балла)

Нужно дообучить модель **генерировать Python-код**. Используй предыдущие примеры как основу.

Точные требования:

- **Датасет:** [codeparrot-clean](https://huggingface.co/datasets/codeparrot/codeparrot-clean) или другой набор с Python-кодом. Для упражнения достаточно небольшого подмножества коротких примеров из валидационной части CodeParrot.
- **Предобработка:** выбирай файлы с расширением `.py` (для CodeParrot этот фильтр можно пропустить, поскольку там уже Python).
- **Ограничение длины:** используй первые **512 символов** каждой строки.
- **Адаптеры:** LoRA, как определено выше, **плюс хотя бы один дополнительный вариант**:
  - Адаптер на `lm_head`.
  - Адаптер на MLP-компоненты (`mlp.*`).
  - Обучаемые входные эмбеддинги (нужно настроить потребление памяти).
- **Обучение:** не обязательно обучать до сходимости. При успешной настройке после ~500 шагов модель должна уметь генерировать код через `.generate`. Используй batch size **не меньше 4** (4 × 1 × 512 токенов), а также `gradient_accumulation_steps=4`.

В `peft` есть готовая LoRA, но **для этого задания нужно показать как минимум один полный цикл обучения с собственной реализацией LoRA**.

**Альтернативное задание:** вместо Python можно выбрать другую этичную задачу и датасет, например тексты любимого автора или подкаста. В этом случае приведи примеры того, что модель выучила — и чего не выучила, как в примерах с кодом ниже.

In [ ]:
prompts =  ['', 'import', 'from', 'while', 'try', 'if', 'for', 'torch']  # feel free to add a few more that are not 100% assiciated with Python

# <A WHOLE LOT OF YOUR CODE>
# сгенерируй базовые примеры по выбранным промптам до fine-tuning
# можешь использовать transformers.Trainer или собственный цикл обучения
# после обучения покажи примеры генерации модели: ожидаются фрагменты Python-кода
# красиво выведи примеры для сравнения (например, через pandas или HTML)
# модель с LoRA может генерировать текст тем же способом, что и базовая модель

In [ ]:
# этот шаблон помогает сравнить сгенерированный код в виде таблицы
# можешь представить работу и в другом формате

from IPython.display import HTML, display
table_template = """<table style="border:1px solid black" >
  <tr>
    <th style="text-align: center; border:1px solid black">PROMPT</th>
    <th style="text-align: center; border:1px solid black">BEFORE</th>
    <th style="text-align: center; border:1px solid black">AFTER</th>
  </tr>
{}
</table>"""

row_template = '''  <tr>
    <td style="width:20%; border:1px solid black"><pre align="left">`{}`</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
  </tr>'''

rows = []

for prompt in prompts:
    # замени заглушки в аргументах format()
    rows.append(row_template.format(prompt, "BEFORE FINETUNING", "TO BE GENERATED AFTER FINETUNING"))

display(HTML(table_template.format('\n'.join(rows))))

PROMPT,BEFORE,AFTER
``,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING
`import`,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING
`from`,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING
`while`,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING
`try`,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING
`if`,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING
`for`,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING
`torch`,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING


Если ты дошёл до этого места — поздравляем, практическая работа завершена!

Чтобы разобраться глубже, попробуй реализовать prompt tuning (дополнительные баллы). Варианты описаны в [статье 1](https://arxiv.org/abs/2104.08691) и [статье 2](https://arxiv.org/abs/2101.00190). Оба можно реализовать, передав обучаемые промпты в `model.forward(..., past_key_values=your_prompts)`.

### Дополнительные материалы
- Посттренировочное квантование: https://arxiv.org/abs/2208.07339
- Запуск больших моделей: https://huggingface.co/docs/accelerate/package_reference/big_modeling
- Библиотека адаптеров: https://adapterhub.ml/

### Дополнительно: другие модели

Код ноутбука можно адаптировать для похожих по размеру моделей: [Falcon-7B](https://huggingface.co/tiiuae/falcon-7b), [OPT-6.7B](https://huggingface.co/facebook/opt-6.7b), [BLOOM-7.1B](https://huggingface.co/bigscience/bloom-7b1). Потребуются небольшие изменения:

1. Замени имя модели и в `AutoModelForCausalLM.from_pretrained()`, и в `AutoTokenizer`.
2. В коде prompt tuning замени `model.model.embed_tokens` на путь к матрице эмбеддингов целевой модели. Посмотри структуру через `print(model)`.
3. Измени код добавления LoRA-слоёв, так как названия модулей трансформера в разных архитектурах различаются.